# CPA Screening — End-to-End Runner

Phase 1 Day 1 (RF baseline) + Phase 1 Day 2 (ChemBERTa-2 + LoRA, rank 8, single seed).

Layout:
1. Setup: mount Drive, clone, install base deps
2. Build dataset + audit
3. RF baseline (Day 1) — completes in ~30s
4. **PAUSE POINT 1** if running for the first time
5. Install deep-learning extras (`requirements-deep.txt`) — torch/transformers/peft
6. ChemBERTa-2 + LoRA rank 8, single seed (Day 2) — ~5–15 min on T4, ~1–3 min on H100
7. Display combined results + parity plots
8. Bundle for download
9. **PAUSE POINT 2**

Phase 2 (cluster splits, deep ensembles, conformal calibration, candidate scoring) lives in a separate notebook section that turns on after PAUSE POINT 2 is cleared.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
![ -d cpa-screening ] || git clone https://github.com/cmendoza1031/cpa-screening.git
%cd /content/cpa-screening
!git pull --ff-only

In [ ]:
# Phase 1 Day 1 base deps. NO --quiet so install errors are visible.
!pip install -r requirements.txt

In [ ]:
# Sanity check: fail loudly here rather than silently inside the pipeline.
import importlib
REQUIRED = ['rdkit', 'pubchempy', 'pandas', 'numpy', 'sklearn', 'scipy', 'matplotlib', 'requests', 'pyarrow']
missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
        print(f'  ok  {mod}')
    except ImportError as e:
        print(f'  MISSING  {mod}: {e}')
        missing.append(mod)
if missing:
    raise RuntimeError(f'Missing required modules: {missing}. Re-run the pip install cell, then retry.')
print('\nAll Phase 1 Day 1 deps available.')

## 2. Build the dataset

In [ ]:
# --skip-tox21 because DeepChem isn't in the Day 1 deps; uncomment --skip-fda
# for a fast smoke run without FDA IID candidate scoring.
!python -m src.data --skip-tox21
# !python -m src.data --skip-tox21 --skip-fda

In [ ]:
import json, pathlib
audit = json.loads(pathlib.Path('data/processed/audit.json').read_text())
print(json.dumps(audit, indent=2)[:2000])

## 3. RF baseline (Phase 1 Day 1)

Eval scheme: 70/15/15 for IRI (n=303), 5-fold CV for the small tasks (toxicity n=22, permeability n=16). LOO-CV permeability is reported as an RF-only sanity check (the gap between 5-fold and LOO is informative about scaffold leakage).

In [ ]:
!python -m src.train --model rf --seed 0

In [ ]:
import pandas as pd
rf_df = pd.read_csv('results/results_table.csv')
rf_df

## 4. ChemBERTa-2 + LoRA (Phase 1 Day 2)

Now install the deep-learning extras and run the multi-task ChemBERTa fine-tune. **Only run this section if you have a GPU runtime selected** (Runtime → Change runtime type → GPU). On CPU it will technically run but takes ~30+ min and the gradient noise on small batches makes results unstable.

Model: `DeepChem/ChemBERTa-77M-MLM` (RoBERTa-style, 384 hidden), mean-pool, 3 regression heads (toxicity, permeability, iri), LoRA (rank 8, alpha 16, dropout 0.1) on q/k/v.

Tox21 auxiliary head is OFF for Phase 1 Day 2 to keep the dep surface small (no DeepChem). Phase 2 turns it on.

In [ ]:
# Install Day 2+ deep-learning extras. NOTE: no -U. -U would force-upgrade
# torch to 2.11+cu130 while torchvision stays at +cu128 (Colab ships them
# paired by CUDA version), and torchvision._check_cuda_version() then
# refuses to load -- which cascades into peft/transformers import failures.
# Without -U, pip leaves Colab's torch alone (it already satisfies >=2.1)
# and only upgrades the actual constraint-violator (torchao 0.10 -> 0.16+).
!pip install -r requirements-deep.txt

In [ ]:
# Hard sanity check on the deep-learning environment. If anything below is
# wrong we want to know NOW, not 50 lines into ChemBERTa training. Specifically
# we previously hit: pre-installed torchao 0.10 -> peft LoRA dispatcher
# raises ImportError when probing is_torchao_available().
import importlib.metadata as md
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Switch via Runtime -> Change runtime type.')

REQ_MIN = {'torch': '2.1', 'transformers': '4.40', 'peft': '0.10', 'accelerate': '0.27', 'torchao': '0.16'}
def _v(s):
    return tuple(int(p) for p in s.split('.')[:2] if p.isdigit())
bad = []
for pkg, minv in REQ_MIN.items():
    try:
        got = md.version(pkg)
        ok = _v(got) >= _v(minv)
        print(f"  {'ok ' if ok else 'BAD'}  {pkg:14s} {got} (need >= {minv})")
        if not ok:
            bad.append(f'{pkg} {got} < {minv}')
    except md.PackageNotFoundError:
        bad.append(f'{pkg} not installed')
        print(f'  BAD  {pkg:14s} NOT INSTALLED')
if bad:
    raise RuntimeError(f'Deep-learning deps need fixing: {bad}. Restart runtime + re-run requirements-deep.txt cell.')

# Probe the actual failure path: peft loading a LoRA adapter on RoBERTa.
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
_m = AutoModel.from_pretrained('DeepChem/ChemBERTa-77M-MLM')
_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['query','key','value'], lora_dropout=0.1, bias='none', task_type='FEATURE_EXTRACTION')
_pm = get_peft_model(_m, _cfg)
_pm.print_trainable_parameters()
print('peft LoRA wrap OK')
del _m, _pm  # free memory before training

In [ ]:
# Phase 1 Day 2 single-seed rank-8 ChemBERTa+LoRA training.
# 5-fold CV across all compounds for direct apples-to-apples comparison
# with the RF baseline (which evaluates the small Higgins tasks via 5-fold CV).
# Trains 5 fresh models, each holding out 1/5 of compounds; OOF predictions
# are aggregated for per-task metrics.
#
# Defaults: 30 epochs/fold, batch 32, lr 5e-5, early-stop patience 5,
# bf16 autocast on supported GPUs (A100/H100/Blackwell) -- fp32 fallback on T4.
!python -m src.train --model chemberta --lora-rank 8 --seed 0 --cv --cv-folds 5

In [ ]:
import pandas as pd
all_df = pd.read_csv('results/results_table.csv')
# Show side-by-side: RF + ChemBERTa
all_df.sort_values(['task', 'split', 'model']).reset_index(drop=True)

In [ ]:
from IPython.display import Image, display
import pathlib
# Show ChemBERTa parity plots (val + test for each task)
for p in sorted(pathlib.Path('results/figures').glob('parity_chemberta_*.png')):
    print(p.name)
    display(Image(str(p)))

## 5. Bundle results for download

In [ ]:
!zip -qr results.zip results data/processed/audit.json && ls -la results.zip

In [ ]:
from google.colab import files
files.download('results.zip')

---
## Phase 2: cluster-aware splits + 5-seed deep ensembles + conformal calibration + FDA Pareto ranking

Phase 2 builds three things on top of Phase 1's working architecture:
1. **Cluster-aware splits** (Tanimoto on Morgan FPs, Butina threshold 0.6) — prevents scaffold leakage between train/test. The gap between random-split and cluster-split metrics is the realistic generalization signal.
2. **5-seed deep ensembles** for both architectures — yields per-compound (mean, std) predictions; std is an epistemic uncertainty proxy.
3. **Split-conformal calibration** of 95% prediction intervals via the OOF residual quantile, with empirical coverage measured on the held-out folds.

Then `src.score_candidates` trains a separate full-data ensemble (no held-out), predicts on the ~1.8k FDA-IID candidates that pass the CPA-like filter, and Pareto-ranks the top-20.

In [ ]:
# RF cluster-split 5-seed ensemble (cheap; ~30s)
!python -m src.train --model rf --seed 0 --n-seeds 5 --split-mode cluster --cv-folds 5

In [ ]:
# ChemBERTa cluster-split 5-seed ensemble (5 seeds * 5 folds = 25 trainings, ~12-15 min on Blackwell)
!python -m src.train --model chemberta --lora-rank 8 --seed 0 --n-seeds 5 --split-mode cluster --cv --cv-folds 5

In [ ]:
# Show all results so far (RF and ChemBERTa, random and cluster, ensemble and single-seed)
import pandas as pd
df = pd.read_csv('results/results_table.csv')
df.sort_values(['task', 'scheme', 'model']).reset_index(drop=True)

### FDA IID candidate scoring + Pareto top-20

Trains a second ensemble (no held-out) on all CPA labels for each architecture, predicts on the FDA-IID candidate pool, applies the OOF-derived q95 from above for 95% prediction intervals, and Pareto-ranks. ChemBERTa is the primary architecture for the headline ranking; RF predictions are also saved.

Compute: ~5 min on Blackwell for ChemBERTa scoring (5 seeds, no CV) + RF nearly free.

In [ ]:
# Use --architecture both to get both RF and ChemBERTa predictions (ChemBERTa primary)
!python -m src.score_candidates --architecture both --n-seeds 5 --top-k 20

In [ ]:
import pandas as pd
top = pd.read_csv('results/candidates/top20.csv')
show_cols = ['ingredient_name', 'cas', 'pareto', 'composite_score',
             'toxicity_mean', 'toxicity_std',
             'permeability_mean', 'permeability_std',
             'iri_mean', 'iri_std']
top[show_cols]

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('results/figures').glob('pareto_3d_*.png')):
    print(p)
    display(Image(str(p)))

In [ ]:
# Re-bundle everything for download
!zip -qr results.zip results data/processed/audit.json && ls -la results.zip
from google.colab import files
files.download('results.zip')

---
## PAUSE POINT 4 (final pause)

Eyeball the top-20 list:
- Are the candidates chemically plausible CPAs? (Small, polar, H-bond-capable molecules with track records of biocompatibility.)
- Any obvious garbage that should be filtered? (Polymers, pigments, biologics that snuck through the CPA-like filter.)
- Any of them already known cryoprotectants the model is rediscovering?
- Any anti-correlation with literature toxicity? (e.g. ethanolamine should be toxic per Higgins; if the model rates it favorably that's a known model error.)

Send the developer the zip + your read on the top-20 chemistry. They'll write the README's Top-20 candidates table + Limitations section, then we're at Phase 3 (final docs + DESIGN_DOC.md).

Phase 2 deliverables checklist:
- [ ] cluster-aware split numbers (RF + ChemBERTa) in results_table.csv
- [ ] random-split single-seed numbers from earlier (already there)
- [ ] 5-seed ensemble q95 + empirical coverage per task per architecture
- [ ] FDA top-20 csv
- [ ] 3D Pareto plot
- [ ] all_scored.csv (full ranking for follow-up)